In [9]:
!pip install catboost

In [10]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

### конфиг

In [11]:
SEED = 42
BETA = 0.5

DATA_DIR = Path('data')
TRAIN_PATH = DATA_DIR / 'train.jsonl'
TEST_PATH = DATA_DIR / 'test.jsonl'

OUT_DIR = Path('result')
OUT_DIR.mkdir(exist_ok=True)

RESULT_PATH = Path('result.csv')

random.seed(SEED)
np.random.seed(SEED)

## eda
**Текст, выделенный полужирным шрифтом**

In [ ]:
def read_jsonl(path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clair_obscur(text):
    if not text:
        return {
            'chars': 0,
            'words': 0,
            'q_count': 0,
            'e_count': 0,
            'upper_ratio': 0.0,
        }

    letters = [ch for ch in text if ch.isalpha()]
    upper_ratio = sum(ch.isupper() for ch in letters) / len(letters) if letters else 0.0

    return {
        'chars': len(text),
        'words': len(text.split()),
        'q_count': text.count('?'),
        'e_count': text.count('!'),
        'upper_ratio': upper_ratio,
    }

def build_df(rows):
    recs = []
    for row in rows:
        msgs = row['messages']
        texts = [m.get('text', '') for m in msgs]
        last_text = texts[-1] if texts else ''
        all_text = ' [SEP] '.join(texts)

        td = np.array([float(m.get('time_diff', 0.0)) for m in msgs], dtype=float)
        rp = np.array([int(m.get('is_reply', 0)) for m in msgs], dtype=int)

        s_last = clair_obscur(last_text)
        s_all = clair_obscur(all_text)

        rec = {
            'session_id': int(row['session_id']),
            'num_messages': len(msgs),
            'sum_time_diff': float(td.sum()) if len(td) else 0.0,
            'mean_time_diff': float(td.mean()) if len(td) else 0.0,
            'max_time_diff': float(td.max()) if len(td) else 0.0,
            'last_time_diff': float(td[-1]) if len(td) else 0.0,
            'sum_is_reply': int(rp.sum()) if len(rp) else 0,
            'last_is_reply': int(rp[-1]) if len(rp) else 0,
            'chars_last': s_last['chars'],
            'words_last': s_last['words'],
            'q_count_last': s_last['q_count'],
            'e_count_last': s_last['e_count'],
            'upper_ratio_last': s_last['upper_ratio'],
            'chars_all': s_all['chars'],
            'words_all': s_all['words'],
            'q_count_all': s_all['q_count'],
            'e_count_all': s_all['e_count'],
            'upper_ratio_all': s_all['upper_ratio'],
        }

        if 'target' in row:
            rec['target'] = int(row['target'])

        recs.append(rec)

    return pd.DataFrame(recs)

train_rows = read_jsonl(TRAIN_PATH)
test_rows = read_jsonl(TEST_PATH)

train_df = build_df(train_rows)
test_df = build_df(test_rows)

print("ok")

ok


In [13]:
FEATURES = [
    'num_messages',
    'sum_time_diff',
    'mean_time_diff',
    'max_time_diff',
    'last_time_diff',
    'sum_is_reply',
    'last_is_reply',
    'chars_last',
    'words_last',
    'q_count_last',
    'e_count_last',
    'upper_ratio_last',
    'chars_all',
    'words_all',
    'q_count_all',
    'e_count_all',
    'upper_ratio_all',
]

X = train_df[FEATURES]
y = train_df['target'].values
X_test = test_df[FEATURES]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

print('lets starat bruh')

lets starat bruh


### тренируем модель

In [ ]:
model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='Logloss',
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8.0,
    random_seed=SEED,
    verbose=200,
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

valid_prob = model.predict_proba(X_valid)[:, 1]

def find_best_threshold(y_true, prob, beta=0.5):
    best_t, best_s = 0.5, -1.0
    thresholds = np.linspace(0.05, 0.95, 181)
    for t in tqdm(thresholds, desc='Threshold search'):
        pred = (prob >= t).astype(int)
        s = fbeta_score(y_true, pred, beta=beta)
        if s > best_s:
            best_t, best_s = float(t), float(s)
    return best_t, best_s

best_t, best_f = find_best_threshold(y_valid, valid_prob, beta=BETA)
print(f'best score F0.5: {best_f:.6f} at threshold={best_t:.4f}')

0:	learn: 0.6719513	test: 0.6718540	best: 0.6718540 (0)	total: 56.5ms	remaining: 1m 52s
200:	learn: 0.4972140	test: 0.5038090	best: 0.5037968 (198)	total: 9.38s	remaining: 1m 23s
400:	learn: 0.4861793	test: 0.5029746	best: 0.5029746 (400)	total: 18.3s	remaining: 1m 12s
600:	learn: 0.4739244	test: 0.5030517	best: 0.5028970 (541)	total: 22.7s	remaining: 52.7s
800:	learn: 0.4620956	test: 0.5035476	best: 0.5028970 (541)	total: 26.5s	remaining: 39.7s
1000:	learn: 0.4521835	test: 0.5039702	best: 0.5028970 (541)	total: 30s	remaining: 29.9s
1200:	learn: 0.4423167	test: 0.5047565	best: 0.5028970 (541)	total: 34.1s	remaining: 22.7s
1400:	learn: 0.4331937	test: 0.5058827	best: 0.5028970 (541)	total: 38.5s	remaining: 16.4s
1600:	learn: 0.4244398	test: 0.5069805	best: 0.5028970 (541)	total: 42s	remaining: 10.5s
1800:	learn: 0.4162721	test: 0.5078168	best: 0.5028970 (541)	total: 45.5s	remaining: 5.02s
1999:	learn: 0.4085175	test: 0.5088150	best: 0.5028970 (541)	total: 50.2s	remaining: 0us

bestTest 

Threshold search:   0%|          | 0/181 [00:00<?, ?it/s]

Best validation F0.5: 0.703192 at threshold=0.5600


### используем модель, дообучив её на валидационной

In [15]:
model_full = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='Logloss',
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8.0,
    random_seed=SEED,
    verbose=False,
)

model_full.fit(X, y)

test_prob = model_full.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_t).astype(int)

result = pd.DataFrame({
    'session_id': test_df['session_id'].astype(int),
    'target': test_pred.astype(int),
})
result.to_csv(RESULT_PATH, index=False)
print('result saved')

result saved


In [16]:
fi = model_full.get_feature_importance()
fi_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': fi,
}).sort_values('importance', ascending=False)

display(fi_df.head(15))

,feature,importance
0,num_messages,64.712784
16,upper_ratio_all,5.639677
12,chars_all,4.355650
4,last_time_diff,3.838646
7,chars_last,3.296697
11,upper_ratio_last,3.128718
2,mean_time_diff,2.770476
3,max_time_diff,2.509024
13,words_all,2.299919
1,sum_time_diff,2.221908
